In [1]:
import pandas as pd

# Read in dataset from articles dataset
df = pd.read_csv("../data/articles.csv")
df

,match_id,source,url,text
0,43,espn,https://www.espn.co.uk/football/report/_/gameI...,Man United v Brighton\nMan United beat Brighto...
1,43,guardian,https://www.theguardian.com/football/2025/oct/...,A Ruben Amorim pirouette and revolving fist-pu...
2,43,skysports,https://www.skysports.com/football/news/11661/...,Man Utd 4-2 Brighton: Matheus Cunha scores fir...
3,44,guardian,https://www.theguardian.com/football/2025/dec/...,From near-total control to collapse to late Br...
4,44,espn,https://www.espn.co.uk/football/match/_/gameId...,Man United v Bournemouth\nFormations & Lineups...
5,44,bbc,https://www.bbc.co.uk/sport/football/live/c4g6...,Goodnightpublished at 22:45 GMT 15 December 20...


In [2]:
# Function to extracts the text from the relevant articles in the dataset for a match

def get_match_articles(match_id):
    return df[df["match_id"]==match_id]

# Test function
get_match_articles(43)

,match_id,source,url,text
0,43,espn,https://www.espn.co.uk/football/report/_/gameI...,Man United v Brighton\nMan United beat Brighto...
1,43,guardian,https://www.theguardian.com/football/2025/oct/...,A Ruben Amorim pirouette and revolving fist-pu...
2,43,skysports,https://www.skysports.com/football/news/11661/...,Man Utd 4-2 Brighton: Matheus Cunha scores fir...


In [3]:
# Funtion that joins the text from all the different articles together

def build_context(match_articles):
    texts = match_articles["text"].tolist()
    return "\n\n".join(texts)

In [4]:
# Function that builds the query that will be fed into the LLM

# Edit OUTPUT FORMAT when a decision has been made about what summary date we want

def build_query(context):
    return f"""
    You are a strict information extraction system. You will return ONLY the valid JSON. Do NOT add commentary around the JSON file.

    TASK:
    Extract structured football match events from the text.

    RULES:
    - Use ONLY information explicitly stated in the text
    - Do NOT guess or infer missing data
    - Do NOT add commentary
    - Do NOT repeat or duplicate events
    - If minute is not explicitly stated, leave it as empty string
    - Return ONLY the valid JSON (no markdown, no backticks, no explanation, no prose prior to the JSON output)

    OUTPUT FORMAT (must match exactly):
    {{
        "home team": ""
        "home team": ""
        "score": ""
        "goals": [
            {{
                "minute": "",
                "player": "",
                "team": ""
            }}
            ],
        "red cards": [
            {{
                "minute": "",
                "player": "",
                "team": ""
            }}
            ],
        "home manager": "",
        "away manager": "",
        "man of the match": "",
        "stadium": ""
    }}

    TEXT:
    {context}
    """.strip()

In [5]:
# Get API key for OpenRouter from .env file

import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

In [6]:
# Python function that interacts with the LLM

# Reasons why OpenRouter was chosen as an API
# Access to multiple models (OpenAI, Google, HuggingFace) using one API
#       Flexibility identified as important at early stage of project
#       Avoids vendor lock in
# Inexpensive

# Reasons why LLama 3 was chosen as AI Model
# Relaible: Widely supported across different APIs (crucial for early stages of project when archeticeture can change)
# Good size (8B --> 8 billion paramters)
# Inexpensive: Works on cheap API tiers
# Fast enough for real time use (could get away with slower times since pipeline will only run infrequently)
# Instruct tuning: Trained specifically to follow user instructions

import requests

def extract_match_info(query):
    response = requests.post(                                       # Send data to server
        url="https://openrouter.ai/api/v1/chat/completions",        # URL where OpenRouter recieves and sends responses
        headers={
            "Authorization": f"Bearer {API_KEY}",                   # Communicates API Key
            "Content-Type": "application/json"                      # Telling API we sending json data
        },
        
        #The actual body of the request
        json={
            "model": "meta-llama/llama-3-8b-instruct",              # The AI model we want to use
            "messages": [
                {"role": "user", "content": query}                # Role being user tells us this is the input. Context is the input text we are giving the AI.
            ]
        }
    )

    data = response.json()                                          # Converts response to a dictionary

    if "choices" in data:
        return data["choices"][0]["message"]["content"]             # Prints output given by the AI. Defaults to the first response if the AI suggests multiple models
    else:
        return f"Error: {data}"

In [7]:
import requests

def extract_match_info(query):
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },

        json={
            "model": "qwen/qwen3-8b",
            "temperature": 0,
            "max_tokens": 800,

            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a strict information extraction system. "
                        "Return ONLY valid JSON. "
                        "Do not infer missing data. "
                        "Do not add commentary."
                    )
                },
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    data = response.json()

    if "choices" in data:
        return data["choices"][0]["message"]["content"]
    else:
        return f"Error: {data}"

In [55]:
# Test that the connection with the LLM model is working

print(extract_match_info("Tell me a joke"))

Why did the scarecrow win an award? 

Because he was outstanding in his field!


In [9]:
# Composes the precedding functions to generate required summary for the sample matches

import json

output = []
for match_id,group in df.groupby("match_id"):
    match_articles = get_match_articles(match_id)
    context = build_context(match_articles)
    query = build_query(context)
    LLM_ouput = extract_match_info(query)
    output.append(json.loads(LLM_ouput))

output

[{'home team': 'Manchester United',
  'away team': 'Brighton',
  'score': '4-2',
  'goals': [{'minute': '24',
    'player': 'Matheus Cunha',
    'team': 'Manchester United'},
   {'minute': '34', 'player': 'Casemiro', 'team': 'Manchester United'},
   {'minute': '61', 'player': 'Bryan Mbeumo', 'team': 'Manchester United'},
   {'minute': '96', 'player': 'Bryan Mbeumo', 'team': 'Manchester United'},
   {'minute': '74', 'player': 'Danny Welbeck', 'team': 'Brighton'},
   {'minute': '96', 'player': 'Charalampos Kostoulas', 'team': 'Brighton'}],
  'red cards': [],
  'home manager': 'Ruben Amorim',
  'away manager': 'Fabian Hurzeler',
  'man of the match': '',
  'stadium': 'Old Trafford'},
 {'home team': 'Manchester United',
  'away team': 'Bournemouth',
  'score': '4-4',
  'goals': [{'minute': '13', 'player': 'Semenyo', 'team': 'Bournemouth'},
   {'minute': '40', 'player': 'Bruno Fernandes', 'team': 'Manchester United'},
   {'minute': '44', 'player': 'Matheus Cunha', 'team': 'Manchester United

In [10]:
# Saves output from LLM query into a json file

import json
with open("../data/match.json","w") as f:
    json.dump(output,f,indent=4)          # indent=4 means each indentation level in the json file is presented with 4 spaces for readibility